# Feature Pyramid Networks (FPN) for Multi-Scale Object Detection

## 1. Multi-Scale Representation & Scale Invariance
Recognizing objects at vastly different scales (e.g., a 16-pixel distant pedestrian vs a 400-pixel foreground bus) is a classical challenge in computer vision.
- **Shallow layers**: High spatial resolution, but low-level semantics (edges, textures). Poor for categorization.
- **Deep layers**: Low spatial resolution (subsampled by stride 32), but rich semantic features. Poor for precise boundary localization.

### The FPN Architecture (Lin et al., CVPR 2017)
FPN constructs an in-network feature pyramid with high semantics at all levels via two pathways:
1. **Bottom-Up Pathway**: Standard feedforward convolutional backbone (e.g. ResNet) generating feature maps $\{C_2, C_3, C_4, C_5\}$ at strides $\{4, 8, 16, 32\}$.
2. **Top-Down Pathway**: Upsamples coarser, semantically stronger feature maps by a factor of $2\times$ via nearest-neighbor upsampling.
3. **Lateral Connections**: Merges bottom-up maps $C_i$ with top-down maps using a $1 \times 1$ conv to equalize channel dimensions:

$$P_i = \operatorname{Conv}_{1\times1}(C_i) + \operatorname{Upsample}_{2\times}(P_{i+1})$$

A final $3 \times 3$ convolution is applied to each $P_i$ to eliminate upsampling aliasing artifacts, yielding pyramid levels $\{P_2, P_3, P_4, P_5\}$.

In [ ]:
import numpy as np

class SimulatedFPN:
    """
    Simulates Feature Pyramid Network (FPN) lateral connections and top-down feature fusion.
    """
    def __init__(self, out_channels=64):
        self.out_channels = out_channels

    def nearest_neighbor_2x(self, x):
        """Upsamples spatial dimensions [H, W] by 2x."""
        return np.repeat(np.repeat(x, 2, axis=1), 2, axis=2)

    def conv_1x1(self, x, in_c, out_c):
        """Simulates 1x1 convolution across channel dimension."""
        # x: [C, H, W] -> [out_c, H, W]
        H, W = x.shape[1], x.shape[2]
        return np.ones((out_c, H, W), dtype=np.float32) * np.mean(x)

    def forward(self, C2, C3, C4, C5):
        """
        Bottom-up inputs:
        C2: [64,  64, 64] (stride 4)
        C3: [128, 32, 32] (stride 8)
        C4: [256, 16, 16] (stride 16)
        C5: [512,  8,  8] (stride 32)
        """
        # P5 is simply 1x1 conv of C5
        P5 = self.conv_1x1(C5, 512, self.out_channels)
        
        # P4 = 1x1(C4) + upsample(P5)
        P4 = self.conv_1x1(C4, 256, self.out_channels) + self.nearest_neighbor_2x(P5)
        
        # P3 = 1x1(C3) + upsample(P4)
        P3 = self.conv_1x1(C3, 128, self.out_channels) + self.nearest_neighbor_2x(P4)
        
        # P2 = 1x1(C2) + upsample(P3)
        P2 = self.conv_1x1(C2, 64, self.out_channels) + self.nearest_neighbor_2x(P3)
        
        return {"P2": P2, "P3": P3, "P4": P4, "P5": P5}

# Verification
fpn = SimulatedFPN(out_channels=64)
c2 = np.random.randn(64, 64, 64).astype(np.float32)
c3 = np.random.randn(128, 32, 32).astype(np.float32)
c4 = np.random.randn(256, 16, 16).astype(np.float32)
c5 = np.random.randn(512, 8, 8).astype(np.float32)

pyramid = fpn.forward(c2, c3, c4, c5)
print("FPN Output Pyramid Shapes:")
for level, tensor in pyramid.items():
    print(f"  Level {level}: Shape {tensor.shape} (Channels unified to 64)")
assert pyramid["P2"].shape == (64, 64, 64)
assert pyramid["P5"].shape == (64, 8, 8)

## 2. Multi-Scale Anchor Assignment Strategy
In Faster R-CNN with FPN, Region Proposal Network (RPN) anchors of different pixel scales are mapped to specific pyramid levels:
- $P_2$: Area $32^2$ pixels (Small objects)
- $P_3$: Area $64^2$ pixels
- $P_4$: Area $128^2$ pixels
- $P_5$: Area $256^2$ pixels
- $P_6$: Area $512^2$ pixels (Large objects, subsampled from $P_5$)

## 3. High-Yield Interview Questions & Best Practices

### Q1: Why is a $3 \times 3$ convolution applied after the lateral addition in FPN?
**Answer:** Nearest-neighbor $2\times$ upsampling creates blocking (aliasing) artifacts where adjacent $2 \times 2$ pixel patches have identical values. Furthermore, lateral addition blends features from two distinct semantic abstractions. The subsequent $3 \times 3$ convolution serves as an anti-aliasing smoothing filter that homogenizes the fused feature map before passing it to detection heads.

### Q2: What is the architectural difference between FPN and PANet (Path Aggregation Network)?
**Answer:** FPN features a single top-down pathway that transfers high-level semantic knowledge down to low-level feature maps. However, low-level spatial localization signals still have to traverse dozens of layers to reach high pyramid levels. **PANet (CVPR 2018)** adds a **bottom-up path augmentation** immediately following the top-down pyramid, creating a short 4-layer shortcut that propagates sharp low-level edge features back up to $P_5$.